# Iterator Design Pattern 

explained using the classic Playlist of Songs example.

#### The Concept

The Iterator Pattern provides a standard way to loop through a collection of objects (like a List, Stack, or Tree) without exposing how the data is actually stored. 

#### Analogy: 
A TV Remote "Next Channel" button. You just press "Next". You don't care if the TV is storing channels in an Array, a Linked List, or a random Hash Map. You just want the next one.

## The Classic OOP Way (Java-Style)

In the strict OOP approach, we manually create an `Iterator` interface. The collection returns this iterator, and the client uses `has_next()` and `next()` to navigate.

#### THE ITERATOR INTERFACE

In [3]:
from abc import ABC, abstractmethod
from typing import Any

class Iterator(ABC):
    @abstractmethod
    def has_next(self) -> bool:
        pass

    @abstractmethod
    def next(self) -> Any:
        pass

#### THE CONCRETE ITERATOR

In [4]:
from typing import List

class SongIterator(Iterator):
    def __init__(self, songs: List[str]):
        self._songs = songs
        self._index = 0

    def has_next(self) -> bool:
        return self._index < len(self._songs)

    def next(self) -> str:
        if self.has_next():
            song = self._songs[self._index]
            self._index += 1
            return song
        raise Exception("End of Iterator")

#### THE COLLECTION (Aggregate)

In [5]:
from typing import List

class Playlist:
    def __init__(self):
        self._songs: List[str] = []

    def add_song(self, song: str):
        self._songs.append(song)

    def create_iterator(self) -> Iterator:
        # We explicitly return our custom iterator
        return SongIterator(self._songs)

#### CLIENT CODE

In [6]:
def main():
    my_playlist = Playlist()
    my_playlist.add_song("Bohemian Rhapsody")
    my_playlist.add_song("Hotel California")
    my_playlist.add_song("Imagine")

    # The manual Java-style loop
    iterator = my_playlist.create_iterator()
    
    print("--- Java-Style Iteration ---")
    while iterator.has_next():
        song = iterator.next()
        print(f"Playing: {song}")

if __name__ == "__main__":
    main()

--- Java-Style Iteration ---
Playing: Bohemian Rhapsody
Playing: Hotel California
Playing: Imagine


## The Pythonic Way

In Python, the Iterator pattern is baked into the language core.
- `__iter__`: Corresponds to create_iterator.
- `__next__`: Corresponds to next.
- `StopIteration`: Exception raised when done (replaces `has_next` check).

We can implement this manually, or even better, use Generators (`yield`), which creates an iterator in one line of code.

#### THE PYTHONIC COLLECTION

In [7]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Playlist:
    songs: List[str] = field(default_factory=list)

    def add_song(self, song: str):
        self.songs.append(song)

    # METHOD A: The Generator Approach (Most Pythonic)
    # 'yield' automatically pauses function execution and saves state,
    # effectively creating a full Iterator class behind the scenes for you.
    def __iter__(self):
        for song in self.songs:
            yield song

#### CLIENT CODE

In [8]:
def main():
    my_playlist = Playlist()
    my_playlist.add_song("Stairway to Heaven")
    my_playlist.add_song("Smells Like Teen Spirit")
    my_playlist.add_song("Billie Jean")

    print("--- Pythonic Iteration ---")
    
    # Python's 'for' loop automatically calls __iter__ and handles StopIteration
    for song in my_playlist:
        print(f"Playing: {song}")

    # Behind the scenes, Python is doing exactly what the Java example did:
    # it = iter(my_playlist)
    # while True:
    #     try:
    #         print(next(it))
    #     except StopIteration:
    #         break

if __name__ == "__main__":
    main()

--- Pythonic Iteration ---
Playing: Stairway to Heaven
Playing: Smells Like Teen Spirit
Playing: Billie Jean


#### Key Differences

| Feature       | Classic OOP                                | Pythonic                                      |
|---------------|----------------------------------------------|------------------------------------------------|
| Mechanic      | `has_next()` check + `next()` retrieval      | `__next__` method + `StopIteration` exception  |
| Implementation| Requires a separate class (`SongIterator`)   | Just use `yield` inside the collection         |
| Usage         | `while iterator.has_next()`                  | `for item in collection`                       |


#### When to use which?

- **Java Way**: Almost never in Python, unless you are porting legacy code or need specific pointer control (like `iterator.peek()` or `iterator.previous()`).
- **Python Way**: Always. It enables compatibility with all Python built-ins like `list()`, `sorted()`, `map()`, and list comprehensions.

# Iterator Design Pattern 

explained using a complex, real-world scenario: Traversing a Corporate Hierarchy (Tree Structure).

#### The Scenario: Tree Traversal

A company hierarchy is a tree: The CEO manages VPs, VPs manage Directors, and Directors manage Engineers. If you want to print a list of every single employee in the company, you can't just loop a list. You have to traverse a tree (Depth-First Search).
- **Complexity**: The iterator must "flatten" this nested tree structure into a simple linear loop for the client. The client shouldn't care about recursion or stacks.

## The Classic OOP Way (Java-Style)

To implement a Depth-First Iterator manually without recursion (since `next()` is called one step at a time), we need to maintain our own Stack. This is surprisingly complex logic to write manually, which highlights why the pattern is useful (it hides this complexity).

#### THE DATA STRUCTURE (The Tree Node)

In [9]:
from typing import List

class Employee:
    def __init__(self, name: str, role: str):
        self.name = name
        self.role = role
        self.subordinates: List[Employee] = []

    def add_subordinate(self, employee: 'Employee'):
        self.subordinates.append(employee)

#### THE ITERATOR INTERFACE

In [11]:
from abc import ABC, abstractmethod
from typing import Any

class Iterator(ABC):
    @abstractmethod
    def has_next(self) -> bool: pass

    @abstractmethod
    def next(self) -> Any: pass

#### CONCRETE ITERATOR (Complex Logic)

In [12]:
from typing import List

class HierarchyIterator(Iterator):
    """
    Traverses a tree using a Stack to simulate recursion.
    This allows us to pause and resume traversal with every next() call.
    """
    def __init__(self, root: Employee):
        # The stack holds nodes we need to visit
        self._stack: List[Employee] = [root]

    def has_next(self) -> bool:
        return len(self._stack) > 0

    def next(self) -> Employee:
        if not self.has_next():
            raise Exception("End of Iterator")

        # 1. Pop the current node
        current_node = self._stack.pop()

        # 2. Add children to stack (in reverse order so they pop in correct order)
        # We push children so that the next call to next() processes them.
        for child in reversed(current_node.subordinates):
            self._stack.append(child)

        return current_node

#### CLIENT CODE

In [13]:
def main():
    # Setup Hierarchy
    ceo = Employee("Alice", "CEO")
    vp_eng = Employee("Bob", "VP Engineering")
    vp_sales = Employee("Charlie", "VP Sales")
    
    dev1 = Employee("Dave", "Dev")
    dev2 = Employee("Eve", "Dev")
    sales1 = Employee("Frank", "Sales")

    ceo.add_subordinate(vp_eng)
    ceo.add_subordinate(vp_sales)
    vp_eng.add_subordinate(dev1)
    vp_eng.add_subordinate(dev2)
    vp_sales.add_subordinate(sales1)

    # Manual Iteration
    iterator = HierarchyIterator(ceo)
    
    print("--- Java-Style Tree Traversal ---")
    while iterator.has_next():
        emp = iterator.next()
        print(f"{emp.role}: {emp.name}")

if __name__ == "__main__":
    main()

--- Java-Style Tree Traversal ---
CEO: Alice
VP Engineering: Bob
Dev: Dave
Dev: Eve
VP Sales: Charlie
Sales: Frank


## The Pythonic Way (Generators)

In Python, we don't need to manually manage a stack or write a `has_next` class. We use Generators (`yield`). Python's `yield from` syntax allows us to delegate iteration to child nodes recursively. The Python interpreter handles the stack state for us automatically!

#### THE DATA STRUCTURE (Node + Iterator)

In [14]:
from dataclasses import dataclass, field
from typing import List, Iterator

@dataclass
class Employee:
    name: str
    role: str
    # Automatically initialize empty list
    subordinates: List['Employee'] = field(default_factory=list)

    def add(self, emp: 'Employee'):
        self.subordinates.append(emp)

    # MAGIC METHOD: __iter__
    # This turns the Tree into an Iterable.
    def __iter__(self) -> Iterator['Employee']:
        # 1. Yield Self first
        yield self
        
        # 2. Recursive Step: Yield from children
        # 'yield from' automatically loops over the child's iterator
        # and passes the values up to the caller.
        for child in self.subordinates:
            yield from child

    # Optional: A filtered iterator (Strategy logic inside the iterator)
    def developers_only(self):
        if "Dev" in self.role or "Engineering" in self.role:
            yield self
        for child in self.subordinates:
            yield from child.developers_only()

#### CLIENT CODE

In [15]:
def main():
    # Setup Hierarchy
    ceo = Employee("Alice", "CEO")
    vp_eng = Employee("Bob", "VP Engineering")
    vp_sales = Employee("Charlie", "VP Sales")
    
    # Building the tree
    ceo.add(vp_eng)
    ceo.add(vp_sales)
    
    vp_eng.add(Employee("Dave", "Dev"))
    vp_eng.add(Employee("Eve", "Dev"))
    vp_sales.add(Employee("Frank", "Sales"))

    print("--- 1. Pythonic Full Traversal ---")
    # LOOK HOW CLEAN THIS IS:
    # No stacks, no while loops, just a simple for loop.
    for emp in ceo:
        print(f"Visited: {emp.name} ({emp.role})")

    print("\n--- 2. Filtered Traversal (Devs Only) ---")
    for emp in ceo.developers_only():
        print(f"Dev Found: {emp.name}")

if __name__ == "__main__":
    main()

--- 1. Pythonic Full Traversal ---
Visited: Alice (CEO)
Visited: Bob (VP Engineering)
Visited: Dave (Dev)
Visited: Eve (Dev)
Visited: Charlie (VP Sales)
Visited: Frank (Sales)

--- 2. Filtered Traversal (Devs Only) ---
Dev Found: Bob
Dev Found: Dave
Dev Found: Eve


#### Why the Pythonic version is powerful

- `yield from`: This keyword replaces the entire complex logic of the `HierarchyIterator` class in the Java version. It handles the recursion and state pausing automatically.
- **Memory Efficient**: Both versions are **"Lazy"**. They don't generate a list of **1,000** employees in memory. They find the next employee only when the loop asks for it.
- **Composability**: You can easily chain iterators (e.g., `filter()` or `map()`) on top of this.

## Iterator Design Pattern 

applied to a complex data structure: a **LinkedHashMap**.

#### The Scenario: LinkedHashMap

A `LinkedHashMap` combines a Hash Table (for fast lookup) with a Doubly Linked List (to remember insertion order). To iterate over this map, we cannot just loop through the hash buckets (which are unordered). We must traverse the internal Linked List from Head to Tail.

- **Hash Map (Dict)**: Provides `O(1)` access to keys.
- **Doubly Linked List**: Maintains the insertion order of keys.

This requires a bidirectional iterator (one that can go `next()` and `prev()`).

## The Classic OOP Way (Java-Style)

In this approach, we manually build the Doubly Linked List nodes. The `Iterator` is a separate class that holds a pointer to the "current" node and offers explicit navigation methods.

#### THE NODE (Doubly Linked)

In [20]:
from typing import Any

class Entry:
    def __init__(self, key: Any, value: Any):
        self.key = key
        self.value = value
        self.next: Optional['Entry'] = None
        self.prev: Optional['Entry'] = None

#### THE ITERATOR INTERFACE (Bidirectional)

In [21]:
from typing import Optional, Any

class ListIterator:
    def __init__(self, start_node: Optional[Entry]):
        self._current = start_node

    def has_next(self) -> bool:
        return self._current is not None

    def next(self) -> Any:
        if not self._current:
            raise Exception("No more elements")
        
        val = self._current.value
        # Move pointer forward
        self._current = self._current.next
        return val

    def has_prev(self) -> bool:
        return self._current is not None and self._current.prev is not None

    def prev(self) -> Any:
        if not self._current or not self._current.prev:
            raise Exception("No previous elements")
        
        # Move pointer backward
        self._current = self._current.prev
        return self._current.value

#### THE COLLECTION (LinkedHashMap)

In [25]:
from typing import Optional, Any, Dict

class LinkedHashMap:
    def __init__(self):
        self._map: Dict[Any, Entry] = {}
        self._head: Optional[Entry] = None
        self._tail: Optional[Entry] = None

    def put(self, key: Any, value: Any):
        if key in self._map:
            # Update existing
            self._map[key].value = value
            return

        # Create new node
        new_node = Entry(key, value)
        
        # Link it to the list
        if self._head is None:
            self._head = new_node
            self._tail = new_node
        else:
            self._tail.next = new_node
            new_node.prev = self._tail
            self._tail = new_node
        
        # Add to map
        self._map[key] = new_node

    def remove(self, key: Any):
        if key not in self._map:
            return
        
        node = self._map[key]
        
        # Unlink from Linked List
        if node.prev:
            node.prev.next = node.next
        else:
            self._head = node.next # Was head

        if node.next:
            node.next.prev = node.prev
        else:
            self._tail = node.prev # Was tail

        # Remove from map
        del self._map[key]

    def get_iterator(self) -> ListIterator:
        return ListIterator(self._head)
    
    def get_reverse_iterator(self) -> ListIterator:
        # Helper to start from tail (for prev calls)
        # Note: A real bidirectional iterator is complex, 
        # usually you implement one that can go both ways from anywhere.
        iter = ListIterator(None)
        iter._current = self._tail # Hack for demo
        return iter

#### CLIENT CODE

In [26]:
def main():
    lhm = LinkedHashMap()
    lhm.put("A", "Alpha")
    lhm.put("B", "Beta")
    lhm.put("C", "Gamma")

    print("--- Java-Style: Forward ---")
    it = lhm.get_iterator()
    while it.has_next():
        print(it.next())

    # Remove middle element to test linking
    lhm.remove("B") 

    print("--- Java-Style: Forward after Remove ---")
    it = lhm.get_iterator()
    while it.has_next():
        print(it.next())

if __name__ == "__main__":
    main()

--- Java-Style: Forward ---
Alpha
Beta
Gamma
--- Java-Style: Forward after Remove ---
Alpha
Gamma


## The Pythonic Way

In Python, we simplify node management using dataclasses. Instead of writing a manual Iterator class with has_next, we use Magic Methods:
- `__setitem__` (`put`)
- `__delitem__` (`remove`)
- `__iter__` (forward iteration)
- `__reversed__` (backward iteration)

To satisfy your requirement for manual `next()` and `prev()` control, I will implement a Cursor object. This is more Pythonic than a raw iterator because standard Python iterators only go forward.